# ColliderFM Diagnostics Tutorial

This notebook is the guided tour of the current repository. It follows the same path the training code uses:

1. load raw ColliderML calorimeter hits from the dataloader
2. inspect the exact point-view contract used before the model
3. compare base and augmented views
4. batch multiple events the way the trainer sees them
5. run model-backed diagnostics on the compact Panda-style scaffold

The goal is not just to make pretty plots. The goal is to answer:
**is each stage of the pipeline shaped correctly, numerically sane, and consistent with the code in `src/collider_fm/`?**

## Repository Flow

The notebook mirrors the repo structure on purpose:

- `src/collider_fm/data.py` streams ColliderML `calo_hits`
- `src/collider_fm/views.py` converts one event into the calo-only point-view contract `[x, y, z, energy]`
- `src/collider_fm/model.py` defines the compact Panda-style student/teacher scaffold
- `src/collider_fm/diagnostics.py` provides shared helpers used by both this notebook and the batch plotting script
- `scripts/plot_diagnostics.py` is the non-interactive companion to this notebook

The model-backed section requires a CUDA environment because the vendored PTv3 / `spconv` path is GPU-only.

## Setup

This cell handles imports, paths, and plotting defaults. The notebook can be run either from the repo root or from inside `notebooks/`.

In [ ]:
import json
import sys
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from collider_fm.diagnostics import compute_pca, encode_view, load_checkpoint, load_events, tensor_summary, to_numpy
from collider_fm.model import create_small_panda_model, create_training_panda_model
from collider_fm.views import augment_point_view, batch_point_views, build_distillation_views, build_point_view_from_event

plt.style.use('default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False


## Configuration

Edit these values to choose data splits, point caps, device selection, and optional figure saving.

In [ ]:
SEED = 7
DETAIL_SPLIT = 'train[0:1]'
REPRESENTATION_SPLIT = 'train[:10]'
TRAIN_VIEW_EVENT_COUNT = 2
DATASET_TYPE = 'ttbar'
PU_CONFIG = 'pu0'
CACHE_DIR = '/mnt/ceph/users/ewulff/data/hf'
MAX_CALO_HITS = 256
COORD_NOISE_SCALE = 0.5
ENERGY_JITTER_SCALE = 0.01
GLOBAL_CROP_RATIO = 0.9
STUDENT_MASK_FRACTION = 0.4
POINT_DROPOUT = 0.05
POINT_FEATURE_SAMPLE_SIZE = 2000
TOP_K_PROTOTYPES = 8
CHECKPOINT_PATH = None
METRICS_PATH = None
SAVE_OUTPUTS = False
OUTPUT_DIR = PROJECT_ROOT / 'diagnostics' / 'notebook_explorer'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Using device: {DEVICE}')
print(f'Saving outputs: {SAVE_OUTPUTS}')
print(f'Checkpoint path: {CHECKPOINT_PATH}')
print(f'Metrics path: {METRICS_PATH}')
if SAVE_OUTPUTS:
    print(f'Output directory: {OUTPUT_DIR}')


## Shared Helpers

These helpers keep the notebook readable and make the diagnostics sections easier to compare. They also encode a few explicit sanity checks, so the notebook acts like a tutorial and a smoke test at the same time.

In [ ]:
def maybe_save(fig: plt.Figure, name: str) -> None:
    fig.tight_layout()
    if SAVE_OUTPUTS:
        fig.savefig(OUTPUT_DIR / name, dpi=200, bbox_inches='tight')
    plt.show()


def pretty_json(payload: dict[str, Any]) -> None:
    print(json.dumps(payload, indent=2, sort_keys=True))


def preview_tensor(tensor: torch.Tensor, rows: int = 5) -> list[Any]:
    array = to_numpy(tensor[:rows])
    return array.tolist()


def describe_mapping(mapping: dict[str, Any]) -> dict[str, Any]:
    description: dict[str, Any] = {}
    for key, value in mapping.items():
        if isinstance(value, torch.Tensor):
            description[key] = {'shape': list(value.shape), 'dtype': str(value.dtype)}
        else:
            description[key] = type(value).__name__
    return description


def assert_view_contract(view: dict[str, torch.Tensor]) -> dict[str, Any]:
    coord_matches = bool(torch.allclose(view['feat'][:, :3], view['coord']))
    energy_matches = bool(torch.allclose(view['feat'][:, 3], view['energy']))
    unique_source_index = int(torch.unique(view['source_index']).numel())
    unique_patch_id = int(torch.unique(view['patch_id']).numel())
    counts = torch.diff(view['offset'], prepend=view['offset'].new_zeros(1))
    return {
        'view_kind': view.get('view_kind', 'unknown'),
        'num_points': int(view['coord'].shape[0]),
        'offset': view['offset'].tolist(),
        'points_per_event': counts.tolist(),
        'grid_size': float(view['grid_size'].item()),
        'masked_points': int(view['mask'].sum().item()),
        'coord_matches_feat_prefix': coord_matches,
        'energy_matches_feat_suffix': energy_matches,
        'unique_source_index': unique_source_index,
        'unique_patch_id': unique_patch_id,
        'source_index_is_unique': unique_source_index == view['coord'].shape[0],
        'all_values_finite': bool(torch.isfinite(view['coord']).all() and torch.isfinite(view['feat']).all()),
    }


def infer_metrics_path(checkpoint_path: str | None, metrics_path: str | None) -> Path | None:
    if metrics_path is not None:
        return Path(metrics_path)
    if checkpoint_path is None:
        return None
    checkpoint = Path(checkpoint_path)
    run_dir = checkpoint.parent.parent if checkpoint.parent.name == 'checkpoints' else checkpoint.parent
    candidate = run_dir / 'metrics.jsonl'
    return candidate if candidate.exists() else None


def load_metric_records(metrics_path: Path | None) -> list[dict[str, Any]]:
    if metrics_path is None or not metrics_path.exists():
        return []
    records = []
    for line in metrics_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line:
            records.append(json.loads(line))
    return records


def metric_series(records: Sequence[dict[str, Any]], key: str) -> tuple[list[float], list[float]]:
    xs = []
    ys = []
    for record in records:
        if key not in record:
            continue
        xs.append(float(record.get('epoch', len(xs) + 1)))
        ys.append(float(record[key]))
    return xs, ys


def create_model(device: torch.device):
    use_training_model = CHECKPOINT_PATH is not None
    model = create_training_panda_model(device=device) if use_training_model else create_small_panda_model(device=device)
    if CHECKPOINT_PATH is not None:
        checkpoint_artifact = load_checkpoint(model, CHECKPOINT_PATH)
        print('Loaded checkpoint:')
        pretty_json(checkpoint_artifact)
    model.eval()
    return model


def prototype_entropy(probs: torch.Tensor) -> torch.Tensor:
    probs = probs.clamp_min(1e-8)
    return -(probs * probs.log()).sum(dim=-1)


def summarize_model_outputs(outputs: Sequence[dict[str, torch.Tensor]]) -> list[dict[str, Any]]:
    return [
        {
            'view_kind': output['view_kind'],
            'point_features': list(output['point_features'].shape),
            'point_logits': list(output['point_logits'].shape),
            'masked_logits': list(output['masked_logits'].shape),
            'pooled': list(output['pooled'].shape),
            'masked_points': int(output['mask'].sum().item()),
        }
        for output in outputs
    ]


In [ ]:
def plot_raw_geometry(event: dict[str, Any], title: str = 'Raw calorimeter event geometry') -> None:
    calo_hits = event['calo_hits']
    coord = np.stack(
        [to_numpy(calo_hits['z']), to_numpy(calo_hits['x']), to_numpy(calo_hits['y'])],
        axis=1,
    )
    energy = to_numpy(calo_hits['energy'])
    positive_energy = energy[energy > 0]
    if positive_energy.size > 0:
        color_vmin = float(positive_energy.min())
        color_vmax = float(np.quantile(positive_energy, 0.99))
        if color_vmax <= color_vmin:
            color_vmax = float(positive_energy.max())
        if color_vmax <= color_vmin:
            color_vmax = color_vmin * 1.01
        marker_size = np.clip(np.log10(energy / color_vmin + 1.0) * 10.0, 4.0, 60.0)
        norm = LogNorm(vmin=color_vmin, vmax=color_vmax)
    else:
        marker_size = np.full_like(energy, 4.0)
        norm = None

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(
        coord[:, 0],
        coord[:, 1],
        coord[:, 2],
        s=marker_size,
        alpha=0.55,
        c=energy,
        cmap='inferno',
        norm=norm,
    )
    colorbar = fig.colorbar(scatter, ax=ax, shrink=0.7, pad=0.1)
    colorbar.set_label('calorimeter energy')
    ax.set_xlabel('z [mm]')
    ax.set_ylabel('x [mm]')
    ax.set_zlabel('y [mm]')
    ax.set_title(title)
    maybe_save(fig, 'raw_event_geometry.png')


def plot_raw_scalars(event: dict[str, Any], title: str = 'Raw dataloader scalar summaries') -> None:
    calo_hits = event['calo_hits']
    coord = torch.stack([calo_hits['x'], calo_hits['y'], calo_hits['z']], dim=1)
    radius = torch.linalg.norm(coord, dim=1)
    energy = calo_hits['energy']
    z_values = calo_hits['z']

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].hist(to_numpy(energy), bins=40, color='tab:red')
    axes[0].set_title('Energy')
    axes[1].hist(to_numpy(radius), bins=40, color='tab:orange')
    axes[1].set_title('Radius')
    axes[2].hist(to_numpy(z_values), bins=40, color='tab:blue')
    axes[2].set_title('z coordinate')
    axes[3].axis('off')
    axes[3].text(
        0.0,
        0.9,
        '\n'.join(
            [
                f'calo hits: {len(calo_hits["x"])}',
                f'energy min/max: [{energy.min().item():.3f}, {energy.max().item():.3f}]',
                f'radius min/max: [{radius.min().item():.3f}, {radius.max().item():.3f}]',
                f'z min/max: [{z_values.min().item():.3f}, {z_values.max().item():.3f}]',
            ]
        ),
        fontsize=11,
        va='top',
    )
    fig.suptitle(title)
    maybe_save(fig, 'raw_event_scalars.png')


def plot_view_before_model(view: dict[str, torch.Tensor], title: str = 'Point view before model') -> None:
    coord = to_numpy(view['coord'])
    energy = to_numpy(view['energy'])
    unique_patch_id, patch_counts = torch.unique(view['patch_id'], return_counts=True)
    top_k = min(12, int(patch_counts.numel()))
    order = torch.argsort(patch_counts, descending=True)[:top_k]
    patch_labels = [str(int(unique_patch_id[index])) for index in order]
    patch_values = to_numpy(patch_counts[order])

    fig = plt.figure(figsize=(15, 10))
    grid = fig.add_gridspec(2, 2)
    ax_scatter = fig.add_subplot(grid[:, 0], projection='3d')
    scatter = ax_scatter.scatter(
        coord[:, 2], coord[:, 0], coord[:, 1], c=energy, cmap='inferno', s=4, alpha=0.7
    )
    fig.colorbar(scatter, ax=ax_scatter, shrink=0.7, pad=0.1, label='energy')
    ax_scatter.set_xlabel('z')
    ax_scatter.set_ylabel('x')
    ax_scatter.set_zlabel('y')
    ax_scatter.set_title('3D point view colored by energy')

    ax_hist = fig.add_subplot(grid[0, 1])
    ax_hist.hist(energy, bins=40, color='tab:red', alpha=0.8)
    ax_hist.set_title('Energy distribution in the point view')
    ax_hist.set_xlabel('energy')

    ax_patch = fig.add_subplot(grid[1, 1])
    ax_patch.bar(np.arange(top_k), patch_values, color='tab:orange')
    ax_patch.set_xticks(np.arange(top_k))
    ax_patch.set_xticklabels(patch_labels, rotation=45, ha='right')
    ax_patch.set_title('Largest coarse patch occupancies')
    ax_patch.set_ylabel('points')
    fig.suptitle(title)
    maybe_save(fig, 'view_before_model.png')


def plot_augmentations(base_view: dict[str, torch.Tensor], aug_a: dict[str, torch.Tensor], aug_b: dict[str, torch.Tensor]) -> None:
    views = [('base', base_view), ('aug A', aug_a), ('aug B', aug_b)]
    fig = plt.figure(figsize=(18, 6))
    for index, (label, view) in enumerate(views, start=1):
        coord = to_numpy(view['coord'])
        energy = to_numpy(view['energy'])
        ax = fig.add_subplot(1, 3, index, projection='3d')
        ax.scatter(coord[:, 2], coord[:, 0], coord[:, 1], c=energy, cmap='inferno', s=4, alpha=0.7)
        ax.set_title(label)
        ax.set_xlabel('z')
        ax.set_ylabel('x')
        ax.set_zlabel('y')
    fig.suptitle('Base and augmented views')
    maybe_save(fig, 'augmentations.png')


def plot_augmentation_deltas(base_view: dict[str, torch.Tensor], aug_a: dict[str, torch.Tensor], aug_b: dict[str, torch.Tensor]) -> None:
    base_coord = base_view['coord']
    base_energy = base_view['energy']
    deltas = {
        'aug A': {
            'coord': torch.linalg.norm(aug_a['coord'] - base_coord, dim=1),
            'energy': aug_a['energy'] - base_energy,
        },
        'aug B': {
            'coord': torch.linalg.norm(aug_b['coord'] - base_coord, dim=1),
            'energy': aug_b['energy'] - base_energy,
        },
    }
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label, values in deltas.items():
        axes[0].hist(to_numpy(values['coord']), bins=40, alpha=0.6, label=label)
        axes[1].hist(to_numpy(values['energy']), bins=40, alpha=0.6, label=label)
    axes[0].set_title('Coordinate displacement magnitude')
    axes[1].set_title('Energy perturbation')
    for axis in axes:
        axis.legend()
    fig.suptitle('Augmentation deltas relative to the base view')
    maybe_save(fig, 'augmentation_delta.png')


def plot_batch_summary(views: list[dict[str, torch.Tensor]], title: str = 'Representation-sample summary') -> None:
    point_counts = [view['coord'].shape[0] for view in views]
    patch_counts = [int(torch.unique(view['patch_id']).numel()) for view in views]
    energy_sums = [float(view['energy'].sum().item()) for view in views]
    indices = np.arange(len(views))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].bar(indices, point_counts, color='tab:red')
    axes[0].set_title('Points per event')
    axes[0].set_xlabel('event index')
    axes[0].set_ylabel('points')
    axes[1].bar(indices, patch_counts, color='tab:orange')
    axes[1].set_title('Unique patches per event')
    axes[1].set_xlabel('event index')
    axes[2].bar(indices, energy_sums, color='tab:blue')
    axes[2].set_title('Total energy per event')
    axes[2].set_xlabel('event index')
    axes[2].set_ylabel('sum energy')
    fig.suptitle(title)
    maybe_save(fig, 'batch_summary.png')


def jensen_shannon_divergence(prob_a: torch.Tensor, prob_b: torch.Tensor) -> float:
    mean_prob = 0.5 * (prob_a + prob_b)
    js = 0.5 * F.kl_div(prob_a.log(), mean_prob, reduction='sum') + 0.5 * F.kl_div(prob_b.log(), mean_prob, reduction='sum')
    return float(js.item())


def embedding_cosine_similarity(embedding_a: torch.Tensor, embedding_b: torch.Tensor) -> float:
    return float(F.cosine_similarity(embedding_a.reshape(1, -1), embedding_b.reshape(1, -1), dim=1).item())


def plot_top_prototypes(student_probs: list[np.ndarray], teacher_probs: list[np.ndarray], top_k: int = TOP_K_PROTOTYPES) -> None:
    stacked = np.vstack(student_probs + teacher_probs)
    mean_scores = stacked.mean(axis=0)
    top_indices = np.argsort(mean_scores)[-top_k:]
    labels = [str(index) for index in top_indices]
    x = np.arange(len(top_indices))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 5))
    series = [
        ('student A', student_probs[0][top_indices]),
        ('student B', student_probs[1][top_indices]),
        ('teacher A', teacher_probs[0][top_indices]),
        ('teacher B', teacher_probs[1][top_indices]),
    ]
    for index, (label, values) in enumerate(series):
        ax.bar(x + (index - 1.5) * width, values, width=width, label=label)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel('prototype index')
    ax.set_ylabel('probability')
    ax.set_title('Top prototype probabilities for one event')
    ax.legend()
    maybe_save(fig, 'top_prototypes.png')


def plot_view_agreement(base_pooled: torch.Tensor, aug_a_pooled: torch.Tensor, aug_b_pooled: torch.Tensor, student_probs: list[torch.Tensor], teacher_probs: list[torch.Tensor]) -> None:
    cosine_values = {
        'base vs aug A': embedding_cosine_similarity(base_pooled, aug_a_pooled),
        'base vs aug B': embedding_cosine_similarity(base_pooled, aug_b_pooled),
        'aug A vs aug B': embedding_cosine_similarity(aug_a_pooled, aug_b_pooled),
    }
    js_values = {
        'student': jensen_shannon_divergence(student_probs[0], student_probs[1]),
        'teacher': jensen_shannon_divergence(teacher_probs[0], teacher_probs[1]),
    }

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(list(cosine_values.keys()), list(cosine_values.values()), color='tab:green')
    axes[0].set_ylim(0.0, 1.05)
    axes[0].set_title('Embedding cosine similarity')
    axes[0].tick_params(axis='x', rotation=20)
    axes[1].bar(list(js_values.keys()), list(js_values.values()), color='tab:orange')
    axes[1].set_title('Prototype JS divergence')
    fig.suptitle('Agreement across views')
    maybe_save(fig, 'view_agreement.png')


def plot_embedding_pca(embeddings: torch.Tensor, event_labels: list[str]) -> None:
    projected = compute_pca(to_numpy(embeddings), n_components=2)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(projected[:, 0], projected[:, 1], c=np.arange(len(event_labels)), cmap='tab10', s=60)
    for index, label in enumerate(event_labels):
        ax.annotate(label, (projected[index, 0], projected[index, 1]), fontsize=8)
    ax.set_title('Pooled event embedding PCA')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    maybe_save(fig, 'embedding_pca.png')


def plot_point_feature_pca(point_features: torch.Tensor, energy: torch.Tensor, max_points: int = POINT_FEATURE_SAMPLE_SIZE, seed: int = SEED) -> None:
    point_features_np = to_numpy(point_features)
    energy_np = to_numpy(energy)
    if point_features_np.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        selected = np.sort(rng.choice(point_features_np.shape[0], size=max_points, replace=False))
    else:
        selected = np.arange(point_features_np.shape[0])
    projected = compute_pca(point_features_np[selected], n_components=2)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        projected[:, 0],
        projected[:, 1],
        c=energy_np[selected],
        cmap='inferno',
        s=8,
        alpha=0.6,
    )
    fig.colorbar(scatter, ax=ax, label='input energy')
    ax.set_title('Backbone point-feature PCA')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    maybe_save(fig, 'point_feature_pca.png')


def plot_batch_model_stats(pooled_embeddings: torch.Tensor, probs: torch.Tensor) -> None:
    embedding_norms = pooled_embeddings.norm(dim=1)
    entropies = prototype_entropy(probs)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(to_numpy(embedding_norms), bins=20, color='tab:blue', alpha=0.8)
    axes[0].set_title('Embedding norms')
    axes[0].set_xlabel('L2 norm')
    axes[1].hist(to_numpy(entropies), bins=20, color='tab:red', alpha=0.8)
    axes[1].set_title('Prototype entropy')
    axes[1].set_xlabel('entropy')
    fig.suptitle('Batch-level model diagnostics')
    maybe_save(fig, 'batch_model_stats.png')

## Load Events

The detailed event drives the single-event plots. The representation sample is used for batch-level summaries and model diagnostics.

In [ ]:
detail_events = load_events(
    split=DETAIL_SPLIT,
    batch_size=64,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    cache_dir=CACHE_DIR,
)
representation_events = load_events(
    split=REPRESENTATION_SPLIT,
    batch_size=64,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    cache_dir=CACHE_DIR,
)

detail_event = detail_events[0]
training_events = representation_events[:TRAIN_VIEW_EVENT_COUNT]

print(f'Detailed event count loaded: {len(detail_events)}')
print(f'Representation sample size: {len(representation_events)}')
print(f'Training-view event count: {len(training_events)}')

In [ ]:
raw_structure = {
    'top_level_keys': list(detail_event.keys()),
    'calo_hits': describe_mapping(detail_event['calo_hits']),
    'calo_preview': {
        'x_first5': preview_tensor(detail_event['calo_hits']['x']),
        'y_first5': preview_tensor(detail_event['calo_hits']['y']),
        'z_first5': preview_tensor(detail_event['calo_hits']['z']),
        'energy_first5': preview_tensor(detail_event['calo_hits']['energy']),
    },
}
pretty_json(raw_structure)

## Raw Dataloader Diagnostics

These plots show the event exactly as it comes out of the dataloader, before any point-view conversion or augmentation.

In [ ]:
plot_raw_geometry(detail_event)
plot_raw_scalars(detail_event)

raw_summary = {
    'num_calo_hits': int(len(detail_event['calo_hits']['x'])),
    'x': tensor_summary(detail_event['calo_hits']['x']),
    'y': tensor_summary(detail_event['calo_hits']['y']),
    'z': tensor_summary(detail_event['calo_hits']['z']),
    'energy': tensor_summary(detail_event['calo_hits']['energy']),
}
pretty_json(raw_summary)

## Build Point Views

`build_point_view_from_event()` is the bridge from raw dataloader output to model input.

The current contract is:

- `coord`: `[..., 3]` point coordinates
- `feat`: `[..., 4]` with exact columns `[x, y, z, energy]`
- `offset`: cumulative event boundaries for batched views
- `source_index`: points back to the original calo-hit row indices
- `patch_id`: coarse spatial grouping used as future masking / cropping bookkeeping
- `mask`: placeholder boolean mask field for the next SSL stage

In [ ]:
base_view = build_point_view_from_event(
    detail_event,
    device=DEVICE,
    max_calo_hits=MAX_CALO_HITS,
)
aug_a = augment_point_view(
    base_view,
    coord_noise_scale=COORD_NOISE_SCALE,
    feat_noise_scale=ENERGY_JITTER_SCALE,
    crop_keep_ratio=GLOBAL_CROP_RATIO,
    mask_fraction=0.0,
)
aug_b = augment_point_view(
    base_view,
    coord_noise_scale=COORD_NOISE_SCALE,
    feat_noise_scale=ENERGY_JITTER_SCALE,
    crop_keep_ratio=GLOBAL_CROP_RATIO,
    mask_fraction=STUDENT_MASK_FRACTION,
)
representation_views = [
    build_point_view_from_event(event, device=DEVICE, max_calo_hits=MAX_CALO_HITS)
    for event in representation_events
]
distillation_batch = build_distillation_views(
    training_events,
    device=DEVICE,
    max_calo_hits=MAX_CALO_HITS,
    coord_noise_scale=COORD_NOISE_SCALE,
    feat_noise_scale=ENERGY_JITTER_SCALE,
    global_crop_ratio=GLOBAL_CROP_RATIO,
    student_mask_fraction=STUDENT_MASK_FRACTION,
    point_dropout=POINT_DROPOUT,
)

base_view_report = assert_view_contract(base_view)
distillation_batch_report = {
    'base_views': [assert_view_contract(view) for view in distillation_batch['base_views']],
    'student_views': [assert_view_contract(view) for view in distillation_batch['student_views']],
    'teacher_views': [assert_view_contract(view) for view in distillation_batch['teacher_views']],
}
pretty_json({'base_view': base_view_report, 'distillation_batch': distillation_batch_report})


In [ ]:
view_preview = {
    'coord_first5': preview_tensor(base_view['coord']),
    'feat_first5': preview_tensor(base_view['feat']),
    'energy_first5': preview_tensor(base_view['energy']),
    'source_index_first10': preview_tensor(base_view['source_index'], rows=10),
    'patch_id_first10': preview_tensor(base_view['patch_id'], rows=10),
}
pretty_json(view_preview)

In [ ]:
plot_view_before_model(base_view, title='Single-event point view before model')
plot_batch_summary(representation_views, title='Representation-sample point-view summary')

## Augmentation Diagnostics

The current augmentations are still deliberately readable, but they now match the masked-global training setup more closely:

- azimuthal rotation around the beam axis
- coordinate jitter
- multiplicative energy jitter
- optional contiguous cropping
- optional point dropout
- optional coarse patch masking on the student side


In [ ]:
plot_augmentations(base_view, aug_a, aug_b)
plot_augmentation_deltas(base_view, aug_a, aug_b)

augmentation_report = {
    'coord_delta_mean_aug_a': float(torch.linalg.norm(aug_a['coord'] - base_view['coord'], dim=1).mean().item()),
    'coord_delta_mean_aug_b': float(torch.linalg.norm(aug_b['coord'] - base_view['coord'], dim=1).mean().item()),
    'energy_delta_mean_aug_a': float((aug_a['energy'] - base_view['energy']).abs().mean().item()),
    'energy_delta_mean_aug_b': float((aug_b['energy'] - base_view['energy']).abs().mean().item()),
    'all_augmented_values_finite': bool(
        torch.isfinite(aug_a['coord']).all()
        and torch.isfinite(aug_a['energy']).all()
        and torch.isfinite(aug_b['coord']).all()
        and torch.isfinite(aug_b['energy']).all()
    ),
}
pretty_json(augmentation_report)

## Model-Backed Diagnostics

This section checks the compact Panda-style scaffold itself. On a GPU node it will:

- instantiate the student / teacher model
- encode the base and augmented views
- run one true multi-view forward pass with `build_distillation_views()` output
- inspect prototype distributions, view agreement, and batch-level embedding geometry

## Checkpoint And Metrics Diagnostics

If `CHECKPOINT_PATH` points at a real training run, the notebook switches from the compact demo model to the training-sized model and can also inspect `metrics.jsonl`.

Recommended pattern after a SLURM run:

- set `CHECKPOINT_PATH` to `runs/<run_name>/checkpoints/best.pt` or `latest.pt`
- leave `METRICS_PATH = None` to auto-discover `runs/<run_name>/metrics.jsonl`
- rerun the model-backed sections to compare fresh-init vs trained behavior


In [ ]:
model = None
detail_base_encoding = None
detail_aug_a_student = None
detail_aug_b_student = None
detail_aug_a_teacher = None
detail_aug_b_teacher = None
train_student_outputs = None
train_teacher_outputs = None
train_loss = None
metric_records = load_metric_records(infer_metrics_path(CHECKPOINT_PATH, METRICS_PATH))

if DEVICE.type != 'cuda':
    print('CUDA is unavailable. Raw and view diagnostics still work, but the current PTv3 / spconv model path requires a GPU.')
else:
    model = create_model(DEVICE)
    num_params = sum(parameter.numel() for parameter in model.parameters())
    with torch.no_grad():
        detail_base_encoding = encode_view(model, base_view, use_teacher=False)
        detail_aug_a_student = encode_view(model, aug_a, use_teacher=False)
        detail_aug_b_student = encode_view(model, aug_b, use_teacher=False)
        detail_aug_a_teacher = encode_view(model, aug_a, use_teacher=True)
        detail_aug_b_teacher = encode_view(model, aug_b, use_teacher=True)
        train_student_outputs, train_teacher_outputs = model(distillation_batch)
        train_loss = model.distillation_loss(train_student_outputs, train_teacher_outputs)

    model_report = {
        'parameter_count': int(num_params),
        'parameter_count_millions': num_params / 1e6,
        'student_outputs': summarize_model_outputs(train_student_outputs),
        'teacher_outputs': summarize_model_outputs(train_teacher_outputs),
        'one_step_distillation_loss': float(train_loss.item()),
        'base_point_feature_shape': list(detail_base_encoding['point_features'].shape),
        'base_pooled_shape': list(detail_base_encoding['pooled'].shape),
        'base_logits_shape': list(detail_base_encoding['logits'].shape),
        'metric_record_count': len(metric_records),
        'all_model_outputs_finite': bool(
            torch.isfinite(detail_base_encoding['point_features']).all()
            and torch.isfinite(detail_base_encoding['pooled']).all()
            and torch.isfinite(detail_base_encoding['logits']).all()
        ),
    }
    pretty_json(model_report)


In [ ]:
if model is None:
    print('Skipping model-backed plots because CUDA is unavailable.')
else:
    student_probs = [
        F.softmax(detail_aug_a_student['logits'][0], dim=-1),
        F.softmax(detail_aug_b_student['logits'][0], dim=-1),
    ]
    teacher_probs = [
        F.softmax(detail_aug_a_teacher['logits'][0], dim=-1),
        F.softmax(detail_aug_b_teacher['logits'][0], dim=-1),
    ]

    plot_top_prototypes(
        [to_numpy(prob) for prob in student_probs],
        [to_numpy(prob) for prob in teacher_probs],
        top_k=TOP_K_PROTOTYPES,
    )
    plot_view_agreement(
        detail_base_encoding['pooled'][0],
        detail_aug_a_student['pooled'][0],
        detail_aug_b_student['pooled'][0],
        student_probs,
        teacher_probs,
    )

    single_event_model_summary = {
        'student_aug_a_entropy': float(prototype_entropy(student_probs[0].unsqueeze(0))[0].item()),
        'student_aug_b_entropy': float(prototype_entropy(student_probs[1].unsqueeze(0))[0].item()),
        'teacher_aug_a_entropy': float(prototype_entropy(teacher_probs[0].unsqueeze(0))[0].item()),
        'teacher_aug_b_entropy': float(prototype_entropy(teacher_probs[1].unsqueeze(0))[0].item()),
        'base_embedding_norm': float(detail_base_encoding['pooled'][0].norm().item()),
        'aug_a_embedding_norm': float(detail_aug_a_student['pooled'][0].norm().item()),
        'aug_b_embedding_norm': float(detail_aug_b_student['pooled'][0].norm().item()),
    }
    pretty_json(single_event_model_summary)

    if metric_records:
        metrics_preview = metric_records[-min(3, len(metric_records)):]
        print('Recent metric records:')
        pretty_json({'records': metrics_preview})


In [ ]:
if model is None:
    print('Skipping batch model diagnostics because CUDA is unavailable.')
else:
    representation_batch = batch_point_views(representation_views)
    with torch.no_grad():
        representation_encoding = encode_view(model, representation_batch, use_teacher=False)
    representation_probs = F.softmax(representation_encoding['logits'], dim=-1)
    event_labels = [f'event_{index:03d}' for index in range(representation_encoding['pooled'].shape[0])]

    plot_embedding_pca(representation_encoding['pooled'], event_labels)
    plot_point_feature_pca(
        representation_encoding['point_features'],
        representation_batch['energy'],
        max_points=POINT_FEATURE_SAMPLE_SIZE,
        seed=SEED,
    )
    plot_batch_model_stats(representation_encoding['pooled'], representation_probs)

    batch_model_summary = {
        'pooled_embeddings': tensor_summary(representation_encoding['pooled']),
        'point_features': tensor_summary(representation_encoding['point_features']),
        'prototype_entropy_mean': float(prototype_entropy(representation_probs).mean().item()),
        'prototype_entropy_std': float(prototype_entropy(representation_probs).std().item()) if representation_probs.shape[0] > 1 else 0.0,
        'embedding_norm_mean': float(representation_encoding['pooled'].norm(dim=1).mean().item()),
        'embedding_norm_std': float(representation_encoding['pooled'].norm(dim=1).std().item()) if representation_encoding['pooled'].shape[0] > 1 else 0.0,
    }
    pretty_json(batch_model_summary)


## Final Sanity Checklist

A healthy current run should show all of the following:

- non-empty raw calorimeter events with sensible coordinate and energy ranges
- a point-view contract where `feat[:, :3] == coord` and `feat[:, 3] == energy`
- augmentations that perturb the event but keep the overall geometry recognizable
- batched training views with meaningful `offset` boundaries
- finite model activations and a finite distillation loss on a GPU node
- prototype entropies and embedding norms that are non-degenerate

In [ ]:
sanity_report = {
    'raw_event': {
        'num_calo_hits': int(len(detail_event['calo_hits']['x'])),
        'energy_summary': tensor_summary(detail_event['calo_hits']['energy']),
    },
    'base_view': assert_view_contract(base_view),
    'augmentations': {
        'coord_delta_mean_aug_a': float(torch.linalg.norm(aug_a['coord'] - base_view['coord'], dim=1).mean().item()),
        'coord_delta_mean_aug_b': float(torch.linalg.norm(aug_b['coord'] - base_view['coord'], dim=1).mean().item()),
        'energy_delta_mean_aug_a': float((aug_a['energy'] - base_view['energy']).abs().mean().item()),
        'energy_delta_mean_aug_b': float((aug_b['energy'] - base_view['energy']).abs().mean().item()),
    },
    'distillation_batch': {
        'num_base_views': len(distillation_batch['base_views']),
        'student_offsets': [view['offset'].tolist() for view in distillation_batch['student_views']],
        'teacher_offsets': [view['offset'].tolist() for view in distillation_batch['teacher_views']],
    },
}

if model is None:
    sanity_report['model'] = {
        'ran_model_diagnostics': False,
        'reason': 'CUDA unavailable in this session',
    }
else:
    sanity_report['model'] = {
        'ran_model_diagnostics': True,
        'distillation_loss': float(train_loss.item()),
        'student_outputs': summarize_model_outputs(train_student_outputs),
        'teacher_outputs': summarize_model_outputs(train_teacher_outputs),
        'base_embedding_norm': float(detail_base_encoding['pooled'][0].norm().item()),
        'student_aug_a_entropy': float(prototype_entropy(F.softmax(detail_aug_a_student['logits'], dim=-1))[0].item()),
        'student_aug_b_entropy': float(prototype_entropy(F.softmax(detail_aug_b_student['logits'], dim=-1))[0].item()),
        'metric_record_count': len(metric_records),
    }

pretty_json(sanity_report)
